In [3]:
import pickle

with open("base_visages.pkl", "rb") as f:
    base = pickle.load(f)

print(len(base["encodages"]), "encodages")
print("Personnes :", sorted(set(base["noms"])))

14 encodages
Personnes : ['Aaron_Eckhart', 'Aaron_Guiel', 'Aaron_Patterson', 'Aaron_Peirsol', 'Aaron_Pena', 'Aaron_Sorkin', 'Aaron_Tippin', 'Abba_Eban', 'Abbas_Kiarostami', 'Abdel_Aziz_Al-Hakim']


In [5]:
import face_recognition
import os
import pickle
from pathlib import Path

# Chemin vers le dossier LFW (fonctionne sur n'importe quel PC)
dossier_visages_connus = str(Path.home() / "Desktop" / "Visage_connus" / "lfw-deepfunneled" / "lfw-deepfunneled")

encodages_connus = []
noms_connus = []

max_personnes = 10          # nombre de personnes chargées
max_images = 20             # images par personne (plus rapide)
mes_dossiers = ["Aya"]      # ton dossier perso, ignoré s'il n'existe pas

# Les personnes les plus photographiées d'abord
toutes = [p for p in os.listdir(dossier_visages_connus)
          if os.path.isdir(os.path.join(dossier_visages_connus, p))]
toutes.sort(key=lambda n: len(os.listdir(os.path.join(dossier_visages_connus, n))), reverse=True)

personnes = toutes[:max_personnes]
for nom in mes_dossiers:
    if nom in toutes and nom not in personnes:
        personnes.append(nom)

for nom_personne in personnes:

    dossier_personne = os.path.join(dossier_visages_connus, nom_personne)

    print("Traitement :", nom_personne)

    for fichier in os.listdir(dossier_personne)[:max_images]:

        chemin = os.path.join(dossier_personne, fichier)

        # Vérifier que c'est une image
        if not fichier.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        try:
            image = face_recognition.load_image_file(chemin)

            encodage = face_recognition.face_encodings(image)

            if len(encodage) > 0:
                encodages_connus.append(encodage[0])
                noms_connus.append(nom_personne)

            else:
                print("Aucun visage détecté :", fichier)

        except Exception as e:
            print("Erreur avec", fichier, ":", e)


# Sauvegarde de la base
with open("base_visages.pkl", "wb") as f:
    pickle.dump(
        {
            "encodages": encodages_connus,
            "noms": noms_connus
        },
        f
    )


print("\nBase créée avec succès")
print("Nombre d'encodages :", len(encodages_connus))
print("Personnes :", sorted(set(noms_connus)))

c:\Users\OMEN\Desktop\Face Recognation App\.venv\Lib\site-packages\face_recognition_models\__init__.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


Traitement : George_W_Bush
Traitement : Colin_Powell
Traitement : Tony_Blair
Traitement : Donald_Rumsfeld
Traitement : Gerhard_Schroeder
Traitement : Ariel_Sharon
Aucun visage détecté : Ariel_Sharon_0018.jpg
Traitement : Hugo_Chavez
Traitement : Junichiro_Koizumi
Traitement : Jean_Chretien
Traitement : John_Ashcroft

Base créée avec succès
Nombre d'encodages : 199
Personnes : ['Ariel_Sharon', 'Colin_Powell', 'Donald_Rumsfeld', 'George_W_Bush', 'Gerhard_Schroeder', 'Hugo_Chavez', 'Jean_Chretien', 'John_Ashcroft', 'Junichiro_Koizumi', 'Tony_Blair']


In [6]:
from PIL import Image
import streamlit as st 
import face_recognition 
import pickle 
import numpy as np 

st.title ("application de reconnaissance faciale")

with open ("base_visages.pkl" , "rb") as f: 
    base = pickle.load(f) 
    fichier = st.file_uploader("Uploader une photo", type = ["jpg","jpeg", "png"])
    if fichier is not None : 
        image = np.array(Image.open(fichier))
        positions = face_recognition.face_locations(image) 
        encodage = face_recognition.face_encodings(image , positions) 
        st.image(image, caption=f"{len(positions)} visage(s) detect(s)")
        for encodage in encodages : 
            distances = face_recognition.face_detection(base["encodages"], encodage)
            idx = np.argmin(distances)
            if distances[idx] < 0.6 : 
                st.success(f"Identifie : {base['noms'][idx]} (distance : {distances[idx]:.2f})")
            else :
                st.warning("Personnes non reconnue")


2026-10-17 12:06:48.495 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-17 12:06:48.574 
  command:

    streamlit run c:\Users\OMEN\Desktop\Face Recognation App\.venv\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-10-17 12:06:48.574 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-17 12:06:48.574 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-17 12:06:48.574 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-17 12:06:48.574 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-17 12:06:48.574 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-17 12:06:48.574

In [8]:
"""Lancer avec :  streamlit run app_streamlit.py"""
import pickle
from pathlib import Path

import cv2
import face_recognition
import numpy as np
import streamlit as st
from PIL import Image, ImageOps

try:
    import av
    from streamlit_webrtc import VideoProcessorBase, webrtc_streamer
    WEBRTC_OK = True
except ImportError:
    WEBRTC_OK = False

st.set_page_config(page_title="Reconnaissance faciale", layout="centered")
st.title("Application de reconnaissance faciale")


@st.cache_resource
def charger_base():
    chemin = Path(__file__).parent / "base_visages.pkl"
    if not chemin.exists():
        return None
    with open(chemin, "rb") as f:
        return pickle.load(f)


base = charger_base()
if base is None or not base["encodages"]:
    st.error("base_visages.pkl est introuvable ou vide. Crée-la d'abord depuis le notebook.")
    st.stop()

seuil = st.sidebar.slider("Seuil de reconnaissance (plus petit = plus strict)", 0.30, 0.80, 0.60, 0.01)


def identifier(encodage, seuil_):
    distances = face_recognition.face_distance(base["encodages"], encodage)
    idx = int(np.argmin(distances))
    nom = base["noms"][idx] if distances[idx] < seuil_ else "Inconnue"
    return nom, float(distances[idx])


def analyser_rgb(image_rgb, seuil_):
    positions = face_recognition.face_locations(image_rgb)
    encodages = face_recognition.face_encodings(image_rgb, positions)
    return [(pos, *identifier(enc, seuil_)) for pos, enc in zip(positions, encodages)]


def dessiner(img_bgr, resultats, echelle=1):
    for (top, right, bottom, left), nom, dist in resultats:
        top, right, bottom, left = [int(c * echelle) for c in (top, right, bottom, left)]
        couleur = (0, 200, 0) if nom != "Inconnue" else (0, 0, 255)
        cv2.rectangle(img_bgr, (left, top), (right, bottom), couleur, 2)
        cv2.putText(img_bgr, f"{nom} ({dist:.2f})", (left, max(top - 10, 20)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, couleur, 2)
    return img_bgr


def reduire(img, max_cote=1000):
    h, w = img.shape[:2]
    f = max_cote / max(h, w)
    return cv2.resize(img, (int(w * f), int(h * f))) if f < 1 else img


def afficher_resultats(image_rgb):
    image_rgb = reduire(image_rgb)
    resultats = analyser_rgb(image_rgb, seuil)
    annotee = dessiner(cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR), resultats)
    st.image(cv2.cvtColor(annotee, cv2.COLOR_BGR2RGB), caption=f"{len(resultats)} visage(s) détecté(s)")
    if not resultats:
        st.info("Aucun visage détecté.")
    for i, (_, nom, dist) in enumerate(resultats, start=1):
        if nom != "Inconnue":
            st.success(f"Visage {i} identifié : {nom} (distance : {dist:.2f})")
        else:
            st.warning(f"Visage {i} : personne non reconnue (distance : {dist:.2f})")


if WEBRTC_OK:
    class Processeur(VideoProcessorBase):
        def __init__(self):
            self.seuil = 0.6
            self.compteur = 0
            self.resultats = []

        def recv(self, frame):
            img = frame.to_ndarray(format="bgr24")
            self.compteur += 1
            if self.compteur % 3 == 0:          # une image sur 3 pour rester fluide
                petite = cv2.resize(img, (0, 0), fx=0.25, fy=0.25)
                rgb = cv2.cvtColor(petite, cv2.COLOR_BGR2RGB)
                self.resultats = analyser_rgb(rgb, self.seuil)
            img = dessiner(img, self.resultats, echelle=4)
            return av.VideoFrame.from_ndarray(img, format="bgr24")


onglet_live, onglet_photo, onglet_fichier = st.tabs(["Caméra en direct", "Photo caméra", "Fichier"])

with onglet_live:
    if not WEBRTC_OK:
        st.warning("Installe d'abord le module : uv pip install streamlit-webrtc")
    else:
        st.write("Clique sur **START**, puis autorise la caméra dans le navigateur.")
        ctx = webrtc_streamer(
            key="reco-live",
            video_processor_factory=Processeur,
            media_stream_constraints={"video": True, "audio": False},
            async_processing=True,
        )
        if ctx.video_processor:
            ctx.video_processor.seuil = seuil

with onglet_photo:
    photo = st.camera_input("Prends une photo")
    if photo is not None:
        afficher_resultats(np.array(Image.open(photo).convert("RGB")))

with onglet_fichier:
    fichier = st.file_uploader("Uploader une photo", type=["jpg", "jpeg", "png"])
    if fichier is not None:
        afficher_resultats(np.array(ImageOps.exif_transpose(Image.open(fichier)).convert("RGB")))

2026-10-17 12:14:29.833 No runtime found, using MemoryCacheStorageManager
2026-10-17 12:14:29.833 No runtime found, using MemoryCacheStorageManager
2026-10-17 12:14:29.841 No runtime found, using MemoryCacheStorageManager
2026-10-17 12:14:29.841 No runtime found, using MemoryCacheStorageManager
2026-10-17 12:14:29.901 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-17 12:14:29.903 No runtime found, using MemoryCacheStorageManager
2026-10-17 12:14:29.931 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-17 12:14:29.932 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-17 12:14:29.932 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-17 12:14:29.932 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


NameError: name '__file__' is not defined

In [9]:
%%writefile app_streamlit.py
import pickle
from pathlib import Path

import cv2
import face_recognition
import numpy as np
import streamlit as st
from PIL import Image, ImageOps

try:
    import av
    from streamlit_webrtc import VideoProcessorBase, webrtc_streamer
    WEBRTC_OK = True
except ImportError:
    WEBRTC_OK = False

st.set_page_config(page_title="Reconnaissance faciale", layout="centered")
st.title("Application de reconnaissance faciale")


@st.cache_resource
def charger_base():
    chemin = Path(__file__).parent / "base_visages.pkl"
    if not chemin.exists():
        return None
    with open(chemin, "rb") as f:
        return pickle.load(f)


base = charger_base()
if base is None or not base["encodages"]:
    st.error("base_visages.pkl est introuvable ou vide. Crée-la d'abord depuis le notebook.")
    st.stop()

seuil = st.sidebar.slider("Seuil de reconnaissance (plus petit = plus strict)", 0.30, 0.80, 0.60, 0.01)


def identifier(encodage, seuil_):
    distances = face_recognition.face_distance(base["encodages"], encodage)
    idx = int(np.argmin(distances))
    nom = base["noms"][idx] if distances[idx] < seuil_ else "Inconnue"
    return nom, float(distances[idx])


def analyser_rgb(image_rgb, seuil_):
    positions = face_recognition.face_locations(image_rgb)
    encodages = face_recognition.face_encodings(image_rgb, positions)
    return [(pos, *identifier(enc, seuil_)) for pos, enc in zip(positions, encodages)]


def dessiner(img_bgr, resultats, echelle=1):
    for (top, right, bottom, left), nom, dist in resultats:
        top, right, bottom, left = [int(c * echelle) for c in (top, right, bottom, left)]
        couleur = (0, 200, 0) if nom != "Inconnue" else (0, 0, 255)
        cv2.rectangle(img_bgr, (left, top), (right, bottom), couleur, 2)
        cv2.putText(img_bgr, f"{nom} ({dist:.2f})", (left, max(top - 10, 20)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, couleur, 2)
    return img_bgr


def reduire(img, max_cote=1000):
    h, w = img.shape[:2]
    f = max_cote / max(h, w)
    return cv2.resize(img, (int(w * f), int(h * f))) if f < 1 else img


def afficher_resultats(image_rgb):
    image_rgb = reduire(image_rgb)
    resultats = analyser_rgb(image_rgb, seuil)
    annotee = dessiner(cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR), resultats)
    st.image(cv2.cvtColor(annotee, cv2.COLOR_BGR2RGB), caption=f"{len(resultats)} visage(s) détecté(s)")
    if not resultats:
        st.info("Aucun visage détecté.")
    for i, (_, nom, dist) in enumerate(resultats, start=1):
        if nom != "Inconnue":
            st.success(f"Visage {i} identifié : {nom} (distance : {dist:.2f})")
        else:
            st.warning(f"Visage {i} : personne non reconnue (distance : {dist:.2f})")


if WEBRTC_OK:
    class Processeur(VideoProcessorBase):
        def __init__(self):
            self.seuil = 0.6
            self.compteur = 0
            self.resultats = []

        def recv(self, frame):
            img = frame.to_ndarray(format="bgr24")
            self.compteur += 1
            if self.compteur % 3 == 0:          # une image sur 3 pour rester fluide
                petite = cv2.resize(img, (0, 0), fx=0.25, fy=0.25)
                rgb = cv2.cvtColor(petite, cv2.COLOR_BGR2RGB)
                self.resultats = analyser_rgb(rgb, self.seuil)
            img = dessiner(img, self.resultats, echelle=4)
            return av.VideoFrame.from_ndarray(img, format="bgr24")


onglet_live, onglet_photo, onglet_fichier = st.tabs(["Caméra en direct", "Photo caméra", "Fichier"])

with onglet_live:
    if not WEBRTC_OK:
        st.warning("Installe d'abord le module : uv pip install streamlit-webrtc")
    else:
        st.write("Clique sur **START**, puis autorise la caméra dans le navigateur.")
        ctx = webrtc_streamer(
            key="reco-live",
            video_processor_factory=Processeur,
            media_stream_constraints={"video": True, "audio": False},
            async_processing=True,
        )
        if ctx.video_processor:
            ctx.video_processor.seuil = seuil

with onglet_photo:
    photo = st.camera_input("Prends une photo")
    if photo is not None:
        afficher_resultats(np.array(Image.open(photo).convert("RGB")))

with onglet_fichier:
    fichier = st.file_uploader("Uploader une photo", type=["jpg", "jpeg", "png"])
    if fichier is not None:
        afficher_resultats(np.array(ImageOps.exif_transpose(Image.open(fichier)).convert("RGB")))

Overwriting app_streamlit.py
